## Import the required modules

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any, Dict, Callable
from QMAP_tools.qmap_inspector import QMAPInspector

In [ ]:
# Not required but improved the default layout of plots
import scienceplots
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

## Getting familiar with the database

QMAP produces two databases a "full_db" and a "agg_db" these can either be loaded using pandas/polars and contain the full data set and aggregated full dataset respectively.

```IMPORTANT: DO NOT LOAD THE FULL "full_db" AS REQUIRES ~100GB OF RAM```
Instead aim to use streaming / pre-filtering to make memory requirements manageable as shown below

In [ ]:
qm = QMAPInspector()
# Load the helper class which by default has the standard relative paths of the database files so should work out of the box
# in most cases, however if the data is stored else where just use the init kwargs to set the correct path

In [ ]:
qm.agg_db
# The full aggregated database can be loaded fully into memory and is constructed by aggregating a set of interesting
# statistical features of each boulder such as its area. Each boulder has a unique ID "boulder_id" (u32) which is used
# to reference the boulder especially when doing operations using both the agg_db and full_db.

In [ ]:
# # Inspect the boulder using the render_multi_plot tool
# qm.render_multi_plot(2186534, "TIR detailed_survey sigma band depth 350")

In [ ]:
qm.full_db.filter(pl.col("i") == 1).collect()
# The full database contains all the data needed to make the agg_db and some. It is very important that you do not try load
# the full database into memory so when previewing a standard way is to just filter by pl.col("i") == 1 which restricts the
# database to 1 / 8192 of its full size and therefore can be fully loaded into memory. We are still able to do operations
# on this dataset as you will see later leveraging the LazyFrame.

## Identify the data to agg

In [ ]:
spectral_pairs = [
    ("TIR detailed_survey band depth 350", "TIR detailed_survey tri_num alpha"),
    ("TIR detailed_survey band depth 440", "TIR detailed_survey tri_num alpha"),
    ("TIR detailed_survey slope 1000", "TIR detailed_survey tri_num alpha"),
    ("TIR detailed_survey ratio 1000", "TIR detailed_survey tri_num alpha"),

    ("TIR recona band depth 350", "TIR recona tri_num alpha"),
    ("TIR recona band depth 440", "TIR recona tri_num alpha"),
    ("TIR recona slope 1000", "TIR recona tri_num alpha"),
    ("TIR recona ratio 1000", "TIR recona tri_num alpha"),

    ("TIR reconb band depth 350", "TIR reconb tri_num alpha"),
    ("TIR reconb band depth 440", "TIR reconb tri_num alpha"),
    ("TIR reconb slope 1000", "TIR reconb tri_num alpha"),
    ("TIR reconb ratio 1000", "TIR reconb tri_num alpha"),

    ("VNIR detailed_survey band depth", "VNIR detailed_survey tri_num alpha"),
    ("VNIR detailed_survey reflectance", "VNIR detailed_survey tri_num alpha"),
    ("VNIR detailed_survey slope1 poly", "VNIR detailed_survey tri_num alpha"),
    ("VNIR detailed_survey slope2 poly", "VNIR detailed_survey tri_num alpha"),

    ("VNIR recona band depth", "VNIR recona tri_num alpha"),
    ("VNIR recona reflectance", "VNIR recona tri_num alpha"),
    ("VNIR recona slope1 poly", "VNIR recona tri_num alpha"),
    ("VNIR recona slope2 poly", "VNIR recona tri_num alpha"),

    ("VNIR reconc band depth", "VNIR reconc tri_num alpha"),
    ("VNIR reconc reflectance", "VNIR reconc tri_num alpha"),
    ("VNIR reconc slope1 poly", "VNIR reconc tri_num alpha"),
    ("VNIR reconc slope2 poly", "VNIR reconc tri_num alpha"),
]

## Plan the agg

In [ ]:
from polars.expr.expr import Expr


weighted_exprs: list[Expr] = []

for spec, alpha in spectral_pairs:

    # If null then now weight
    weight: Expr = (
        pl.when(pl.col(spec).is_not_null())
        .then(1 / pl.col(alpha))
        .otherwise(0)
    )

    weighted_exprs.extend([
        (
            (pl.col(spec) * weight)
            .sum()
            .alias(f"{spec} wx")
        ),

        (
            weight
            .sum()
            .alias(f"{spec} w")
        ),

        (
            (pl.col(spec).pow(2) * weight)
            .sum()
            .alias(f"{spec} wx2")
        ),
    ])


# unique intersections
for alpha in set(alpha for _, alpha in spectral_pairs):

    tri_num_col = alpha.replace(" alpha", "")

    weighted_exprs.append(
        pl.col(tri_num_col)
        .unique()
        .len()
        .alias(f"numb_intersect_{tri_num_col}")
    )

## Run and merge the agg

In [ ]:
from tqdm import tqdm
from functools import reduce

agg_dfs = []

for expr in tqdm(weighted_exprs, desc="Aggregating features"):
    agg_df = (
        qm.full_db
        .filter(pl.col("boulder_id").is_not_null())
        .group_by("boulder_id")
        .agg(expr)
        .collect(engine="streaming")
    )
    
    agg_dfs.append(agg_df)

# Join all individual aggregations on boulder_id
agg_plus_df = reduce(
    lambda left, right: left.join(
        right,
        on="boulder_id",
        how="inner"
    ),
    agg_dfs
).join(
    qm.agg_db,
    on = "boulder_id"
)

## Calculate the final results

In [ ]:
weighted_final_exprs = []

for spec, _ in spectral_pairs:

    mean = (
        pl.col(f"{spec} wx")
        /
        pl.col(f"{spec} w")
    )

    weighted_final_exprs.extend([
        mean.alias(f"{spec} Weighted"),

        (
            (
                pl.col(f"{spec} wx2")
                /
                pl.col(f"{spec} w")
            )
            - mean.pow(2)
        )
        .sqrt()
        .alias(f"{spec} Weighted std")
    ])


spec_agg = agg_plus_df.with_columns(
    weighted_final_exprs
).drop(
    [
        col
        for col in agg_plus_df.columns
        if col.endswith((" wx", " w", " wx2"))
    ]
)

spec_agg

## Plot the results

In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

export_folder = Path(".plots/scatter_weighted")
export_folder.mkdir(exist_ok=True, parents=True)

# Columns to plot
columns_of_interest = [
    # Weighted spectral quantities
    *[
        c
        for c in spec_agg.columns
        if c.endswith(" Weighted")
    ],

    # Boulder properties
    "Gamma",
    "Tau",

    # Positions / bounds
    "center_x",
    "center_y",
    "center_z",
]

# Remove duplicates while preserving order
columns_of_interest = list(dict.fromkeys(columns_of_interest))

for x_item, y_item in combinations(columns_of_interest, 2):

    try:
        export_path = export_folder / f"{x_item} vs {y_item}.png"

        if export_path.exists():
            print(f"Already done {x_item} vs {y_item}")
            continue

        spec_agg_filter = (
            spec_agg
            .filter(
                pl.col("Gamma") < 15,
                pl.col("center_x").abs() < 1,
                pl.col("center_y").abs() < 1,
                pl.col("center_z").abs() < 1,
                pl.col(x_item).is_not_nan(),
                pl.col(y_item).is_not_nan(),
            )
        )

        # Skip empty plots
        if spec_agg_filter.height == 0:
            print(f"Skipping {x_item} vs {y_item} (no valid points)")
            continue

        plt.figure(figsize=(6, 4.5))

        plt.scatter(
            spec_agg_filter[x_item],
            spec_agg_filter[y_item],
            s=1,
            # alpha=0.5,
            linewidths=0,
            rasterized=True,
        )

        plt.xlabel(x_item)
        plt.ylabel(y_item)

        plt.tick_params(
            direction="in",
            top=True,
            right=True,
        )

        plt.minorticks_on()
        plt.tight_layout()

        plt.savefig(
            export_path,
            bbox_inches="tight",
        )
        plt.close()

        print(f"Done {x_item} vs {y_item}")

    except Exception as e:
        print(f"Failed for {x_item} vs {y_item}: {e}")
        plt.close()
        continue